# EMSC SeismicPortal — Real-Time Earthquake Stream

This notebook connects to the [EMSC SeismicPortal WebSocket](https://www.seismicportal.eu/realtime.html) and consumes real-time earthquake events.


EMSC: European-Mediterranean Seismological Centre

**WebSocket endpoint:** `wss://www.seismicportal.eu/standing_order/websocket`

Each message is a JSON object with an `action` field (`create` or `update`) and a GeoJSON `data` field containing earthquake properties like magnitude, location, depth, and time.

> ⚠️ Earthquakes are detected every ~5–15 minutes globally. Be patient — the stream is real but low-volume by nature.

Sources: https://github.com/EMSC-CSEM/webservices101

## 1. Install dependencies

In [1]:
%pip install websockets pandas ipywidgets --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [3]:
import asyncio
import json
import signal
from datetime import datetime, timezone

import pandas as pd
import websockets
from IPython.display import clear_output, display

## 3. Configuration

In [4]:
WEBSOCKET_URI = "wss://www.seismicportal.eu/standing_order/websocket"

# How long to listen (seconds). Set to None to run indefinitely.
LISTEN_DURATION_SECONDS = 300  # 5 minutes

# Only display earthquakes above this magnitude (set to 0 to see all)
MIN_MAGNITUDE = 0.0

# Ping interval to keep the connection alive (seconds)
PING_INTERVAL = 15

## 4. Message parser

Each WebSocket message looks like this:
```json
{
  "action": "create",
  "data": {
    "type": "Feature",
    "geometry": { "type": "Point", "coordinates": [lon, lat, depth] },
    "properties": {
      "mag": 4.2,
      "magtype": "mb",
      "time": "2025-03-05T12:00:00.0Z",
      "depth": 7.0,
      "lat": 40.1,
      "lon": 28.5,
      "flynn_region": "TURKEY",
      "auth": "NEIC",
      "unid": "20250305_0000090"
    }
  }
}
```

In [5]:
def parse_message(raw: str) -> dict | None:
    """Parse a raw WebSocket message into a flat dict."""
    try:
        msg = json.loads(raw)
        action = msg.get("action", "unknown")
        props = msg["data"]["properties"]
        coords = msg["data"]["geometry"]["coordinates"]

        return {
            "received_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
            "action": action,
            "event_id": props.get("unid", "-"),
            "time": props.get("time", "-"),
            "magnitude": props.get("mag", None),
            "mag_type": props.get("magtype", "-"),
            "depth_km": props.get("depth", None),
            "lat": coords[1] if len(coords) > 1 else None,
            "lon": coords[0] if len(coords) > 0 else None,
            "region": props.get("flynn_region", "-"),
            "source": props.get("auth", "-"),
        }
    except (KeyError, json.JSONDecodeError) as e:
        print(f"[parse error] {e} — raw: {raw[:100]}")
        return None

## 5. Consumer — collect events into a list

In [6]:
events = []  # All received events are stored here

async def consume(duration_seconds=None, min_magnitude=0.0):
    """Connect to the WebSocket and consume earthquake events."""
    print(f"Connecting to {WEBSOCKET_URI} ...")
    print(f"Listening for {'∞' if duration_seconds is None else str(duration_seconds) + 's'} "
          f"| Min magnitude: {min_magnitude}")
    print("─" * 60)

    start = asyncio.get_event_loop().time()

    try:
        async with websockets.connect(
            WEBSOCKET_URI,
            ping_interval=PING_INTERVAL,
            ping_timeout=30,
        ) as ws:
            print("✅ Connected. Waiting for earthquake events...\n")

            async for raw_message in ws:
                event = parse_message(raw_message)

                if event is None:
                    continue

                # Filter by magnitude
                mag = event.get("magnitude")
                if mag is not None and mag < min_magnitude:
                    continue

                events.append(event)

                # Pretty print the event
                action_icon = "🆕" if event["action"] == "create" else "🔄"
                print(
                    f"{action_icon} [{event['received_at']}] "
                    f"M{mag} {event['mag_type']} | "
                    f"Depth: {event['depth_km']} km | "
                    f"{event['region']} | "
                    f"Source: {event['source']}"
                )

                # Stop after duration
                if duration_seconds is not None:
                    elapsed = asyncio.get_event_loop().time() - start
                    if elapsed >= duration_seconds:
                        print(f"\n⏱ Duration of {duration_seconds}s reached. Stopping.")
                        break

    except websockets.exceptions.ConnectionClosedError as e:
        print(f"\n⚠️  Connection closed: {e}")
    except asyncio.CancelledError:
        print("\n⛔ Cancelled.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        print(f"\nTotal events received: {len(events)}")

## 6. Start consuming

Run the cell below to start listening. It will run for `LISTEN_DURATION_SECONDS` (default: 5 minutes).

**Interrupt the kernel** (■ button) to stop early at any time.

In [7]:
events.clear()  # Reset from previous runs

await consume(
    duration_seconds=LISTEN_DURATION_SECONDS,
    min_magnitude=MIN_MAGNITUDE,
)

Connecting to wss://www.seismicportal.eu/standing_order/websocket ...
Listening for 300s | Min magnitude: 0.0
────────────────────────────────────────────────────────────
✅ Connected. Waiting for earthquake events...

🔄 [2026-04-24 09:28:56 UTC] M1.5 ml | Depth: 7.0 km | WESTERN TURKEY | Source: AFAD
🔄 [2026-04-24 09:29:01 UTC] M4.6 mb | Depth: 35.0 km | MOLUCCA SEA | Source: NEIC

⛔ Cancelled.

Total events received: 2


## 7. Inspect collected events as a DataFrame

In [8]:
if not events:
    print("No events collected yet. Run section 6 first and wait for earthquakes.")
else:
    df = pd.DataFrame(events)
    df["time"] = pd.to_datetime(df["time"])
    df = df.sort_values("time", ascending=False).reset_index(drop=True)
    print(f"Total events: {len(df)}")
    display(df)

Total events: 2


,received_at,action,event_id,time,magnitude,mag_type,depth_km,lat,lon,region,source
0,2026-04-24 09:28:56 UTC,update,20260424_0000150,2026-04-24 08:42:50+00:00,1.5,ml,7.0,36.9153,29.0892,WESTERN TURKEY,AFAD
1,2026-04-24 09:29:01 UTC,update,20260424_0000156,2026-04-24 08:41:38.770000+00:00,4.6,mb,35.0,1.5616,126.6602,MOLUCCA SEA,NEIC


## 8. Basic statistics

In [9]:
if not events:
    print("No events collected yet.")
else:
    df = pd.DataFrame(events)

    print("=== Magnitude stats ===")
    print(df["magnitude"].describe().round(2))

    print("\n=== Top 5 regions ===")
    print(df["region"].value_counts().head())

    print("\n=== Events by action ===")
    print(df["action"].value_counts())

    print("\n=== Events by source ===")
    print(df["source"].value_counts())

=== Magnitude stats ===
count    2.00
mean     3.05
std      2.19
min      1.50
25%      2.28
50%      3.05
75%      3.82
max      4.60
Name: magnitude, dtype: float64

=== Top 5 regions ===
region
WESTERN TURKEY    1
MOLUCCA SEA       1
Name: count, dtype: int64

=== Events by action ===
action
update    2
Name: count, dtype: int64

=== Events by source ===
source
AFAD    1
NEIC    1
Name: count, dtype: int64


## 9. (Optional) Save events to CSV

In [ ]:
if events:
    output_path = "earthquake_events.csv"
    df = pd.DataFrame(events)
    df.to_csv(output_path, index=False)
    print(f"Saved {len(df)} events to {output_path}")
else:
    print("No events to save.")

## 10. (Optional) Continuous listener — runs until interrupted

Use this cell if you want to listen indefinitely. **Interrupt the kernel to stop.**

In [ ]:
# Uncomment to run indefinitely
# events.clear()
# await consume(duration_seconds=None, min_magnitude=MIN_MAGNITUDE)